# Procesamiento de Datos (Descarga, EDA y Limpieza)

Este notebook unifica la descarga del dataset bruto, su exploración inicial y el filtrado necesario para crear particiones robustas para los modelos de recomendación.

## Descarga del Dataset

In [20]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("edusanketdk/electronics")
print("Ruta a los archivos descargados:", path)

Ruta a los archivos descargados: C:\Users\User\.cache\kagglehub\datasets\edusanketdk\electronics\versions\2


## Exploración Rápida

In [21]:
csv_path = os.path.join(path, "electronics.csv")
if not os.path.exists(csv_path):
    csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
    csv_path = os.path.join(path, csv_files[0]) if csv_files else csv_path

df = pd.read_csv(csv_path)
print("--- Dimensiones del Dataset ---")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

print("\n--- Columnas y Tipos de Datos ---")
df.info()

print("\n--- Primeras 5 filas del Dataset ---")
display(df.head())

--- Dimensiones del Dataset ---
Filas: 1,292,954
Columnas: 10

--- Columnas y Tipos de Datos ---
<class 'pandas.DataFrame'>
RangeIndex: 1292954 entries, 0 to 1292953
Data columns (total 10 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   item_id     1292954 non-null  int64  
 1   user_id     1292954 non-null  int64  
 2   rating      1292954 non-null  float64
 3   timestamp   1292954 non-null  str    
 4   model_attr  1292954 non-null  str    
 5   category    1292954 non-null  str    
 6   brand       331120 non-null   str    
 7   year        1292954 non-null  int64  
 8   user_attr   174124 non-null   str    
 9   split       1292954 non-null  int64  
dtypes: float64(1), int64(4), str(5)
memory usage: 98.6 MB

--- Primeras 5 filas del Dataset ---


,item_id,user_id,rating,timestamp,model_attr,category,brand,year,user_attr,split
0,0,0,5.0,1999-06-13,Female,Portable Audio & Video,NaN,1999,NaN,0
1,0,1,5.0,1999-06-14,Female,Portable Audio & Video,NaN,1999,NaN,0
2,0,2,3.0,1999-06-17,Female,Portable Audio & Video,NaN,1999,NaN,0
3,0,3,1.0,1999-07-01,Female,Portable Audio & Video,NaN,1999,NaN,0
4,0,4,2.0,1999-07-06,Female,Portable Audio & Video,NaN,1999,NaN,0


## Limpieza de Columnas y Tipos

In [22]:
print("Columnas antes de la limpieza:", df.columns.tolist())

cols_to_drop = ['model_attr', 'user_attr', 'brand', 'year', 'split', 'category']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

df['rating'] = df['rating'].astype(float)
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])

print("\nLimpieza completada.")
print("Columnas resultantes:", df.columns.tolist())
print("Tipos de datos:")
print(df.dtypes)

Columnas antes de la limpieza: ['item_id', 'user_id', 'rating', 'timestamp', 'model_attr', 'category', 'brand', 'year', 'user_attr', 'split']

Limpieza completada.
Columnas resultantes: ['item_id', 'user_id', 'rating', 'timestamp']
Tipos de datos:
item_id               int64
user_id               int64
rating              float64
timestamp    datetime64[us]
dtype: object


## Filtrado de Actividad Mínima

In [ ]:
min_item_ratings = 5
min_user_ratings = 1 

print(f"Filas antes del filtrado: {len(df):,}")

# 1. Filtrado de ítems (una sola pasada)
item_counts = df['item_id'].value_counts()
valid_items = item_counts[item_counts >= min_item_ratings].index
df = df[df['item_id'].isin(valid_items)]

# 2. Filtrado de usuarios (una sola pasada)
user_counts = df['user_id'].value_counts()
valid_users = user_counts[user_counts >= min_user_ratings].index
df = df[df['user_id'].isin(valid_users)]

n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()

if n_users == 0 or n_items == 0:
    print("El filtrado es demasiado estricto y ha eliminado todas las filas.")
    density, sparsity = 0.0, 1.0
else:
    density = len(df) / (n_users * n_items)
    sparsity = 1 - density

print(f"Filas después del filtrado: {len(df):,}")
print(f"Usuarios únicos: {n_users:,}")
print(f"Ítems únicos: {n_items:,}")
print(f"Densidad de la matriz: {density:.6f} (Sparsity: {sparsity:.6f})")


Filas antes del filtrado: 1,292,954
Filas después del filtrado: 1,292,908
Usuarios únicos: 1,157,599
Ítems únicos: 9,541
Densidad de la matriz: 0.000117 (Sparsity: 0.999883)


Notamos que la mayoría de usuarios han valorado algún producto una sola vez. Por lo que decidimos trabajar sin filtrarlos por cantidad de reseñas (mínimo de 1 reseña).

## Particionado del Dataset

In [24]:
from sklearn.model_selection import train_test_split

print("Iniciando particionado (70% Train, 15% Val, 15% Test)...")

# Primer split: 70% train, 30% temp (val + test)
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)

# Segundo split: Dividir el temp en 50% val y 50% test (15% del total cada uno)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

# Garantizar que no haya problemas de Cold-Start en validación y test
train_users = set(train_df['user_id'])
train_items = set(train_df['item_id'])

def filter_known(df_split):
    known_mask = df_split['user_id'].isin(train_users) & df_split['item_id'].isin(train_items)
    return df_split[known_mask], df_split[~known_mask]

val_df_clean, val_removed = filter_known(val_df)
test_df_clean, test_removed = filter_known(test_df)

# Reintegrar los registros removidos al set de entrenamiento
train_df = pd.concat([train_df, val_removed, test_removed]).reset_index(drop=True)
val_df = val_df_clean.reset_index(drop=True)
test_df = test_df_clean.reset_index(drop=True)

print("Particionado completado con garantía de cobertura en el conjunto de entrenamiento.")

Iniciando particionado (70% Train, 15% Val, 15% Test)...
Particionado completado con garantía de cobertura en el conjunto de entrenamiento.


## Exportación

In [25]:
import os

train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)
test_df.to_csv('test.csv', index=False)

print("--- Archivos exportados exitosamente a 'dataset/' ---")
print(f"train.csv: {len(train_df):,} filas")
print(f"val.csv  : {len(val_df):,} filas")
print(f"test.csv : {len(test_df):,} filas")

--- Archivos exportados exitosamente a 'dataset/' ---
train.csv: 1,237,901 filas
val.csv  : 27,519 filas
test.csv : 27,488 filas
